# Character Dialogue/Mood Agent 테스트 (Production Level)

캐릭터 대화 스타일 및 감정 상태 추출 에이전트 테스트 노트북

## 역할: "Dialogue Designer" (대화 디자이너)
- 대화 스타일 (tone, catchphrases)
- 금기 주제 (forbidden_topics)
- **현재 감정** (current_mood) - **일시적 상태**

## Production Features (v2.0)
- ✅ **voice_config**: TTS 파라미터 (pitch, rate, stability)
- ✅ **speech_samples**: Few-shot LLM 프롬프팅용 예시 문장
- ✅ **dialogue_components**: 비언어적 표현 분리 (게임 엔진용)

> **⚠️ 핵심 검증**: `두려움`이 `current_mood.emotion`에 있어야 함

In [1]:
import sys, os, json, asyncio
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))
print(f"Project root: {project_root}")

Project root: c:\jungle\weapon\sto-link-AI-backend


In [2]:
SAMPLE_STORY = """아린은 어두운 숲 한가운데 서 있었다. '누구야?' 아린이 외쳤다. 그녀의 목소리는 단호했지만 약간의 두려움이 섞여 있었다.

'오랜만이군, 아린.' 카엘의 목소리는 차가웠다. 그의 회색 눈동자는 감정을 드러내지 않았다.

'너를 찾고 있었어.' 카엘이 검을 뽑았다. '이번엔 네가 지는 거다.'

(한숨을 쉬며) 아린은 입술을 깨물며 전투 자세를 취했다. '배신자와 할 말은 없어.' 그녀의 심장이 빠르게 뛰기 시작했다."""

def create_base_state(story=SAMPLE_STORY):
    return {"content": story, "completed_agents": [], "errors": [], "messages": []}

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import nest_asyncio; nest_asyncio.apply()
            return loop.run_until_complete(coro)
        return asyncio.run(coro)
    except: return asyncio.run(coro)

## 1. Dialogue/Mood Agent 직접 Import 및 실행

In [3]:
# 직접 모듈 import (LangGraph 의존성 회피)
import importlib.util
spec = importlib.util.spec_from_file_location(
    "dialogue_mood", 
    os.path.join(project_root, "app/agents/extraction/character/dialogue_mood.py")
)
dialogue_mood_module = importlib.util.module_from_spec(spec)

# llm 모듈 먼저 로드
from app.agents.llm import get_structured_llm

spec.loader.exec_module(dialogue_mood_module)
dialogue_mood_extraction_node = dialogue_mood_module.dialogue_mood_extraction_node
print("✅ Dialogue/Mood Agent 모듈 로드 성공")

✅ Dialogue/Mood Agent 모듈 로드 성공


In [4]:
async def test_dialogue_mood():
    print("💬 Dialogue/Mood Agent 테스트 (Production Level)...")
    return await dialogue_mood_extraction_node(create_base_state())

result = run_async(test_dialogue_mood())

# 에러 확인
if result.get('errors'):
    print(f"\n❌ 에러 발생:")
    for err in result.get('errors', []):
        print(f"   {err}")
else:
    dialogue_data = result.get('char_dialogue_mood', {})
    print(f"\n✅ 추출 완료:")
    print(f"   - 캐릭터 수: {len(dialogue_data)}개")
    print(f"   - 이름: {list(dialogue_data.keys())}")

💬 Dialogue/Mood Agent 테스트 (Production Level)...

✅ 추출 완료:
   - 캐릭터 수: 2개
   - 이름: ['아린', '카엘']


## 2. Human-Readable 출력

In [5]:
dialogue_data = result.get('char_dialogue_mood', {})

if not dialogue_data:
    print("❌ 캐릭터 데이터 없음")
else:
    print("="*70)
    print("💬 Dialogue/Mood Data (캐릭터별 대화 스타일 + 감정)")
    print("="*70)

    for name, data in dialogue_data.items():
        dialogue = data.get('dialogue', {})
        mood = data.get('current_mood', {})
        
        print(f"\n🧑 {name}")
        print(f"   대화 스타일:")
        print(f"      tone: {dialogue.get('tone', 'N/A')}")
        print(f"      catchphrases: {dialogue.get('catchphrases', [])}")
        print(f"      forbidden_topics: {dialogue.get('forbidden_topics', [])}")
        
        print(f"   현재 감정:")
        intensity = mood.get('intensity', 0)
        emotion_bar = '█' * intensity + '░' * (10 - intensity)
        print(f"      emotion: {mood.get('emotion', 'N/A')}")
        print(f"      intensity: [{emotion_bar}] {intensity}/10")
        print(f"      trigger: {mood.get('trigger', 'N/A')}")

💬 Dialogue/Mood Data (캐릭터별 대화 스타일 + 감정)

🧑 아린
   대화 스타일:
      tone: 단호
      catchphrases: []
      forbidden_topics: []
   현재 감정:
      emotion: 두려움
      intensity: [██████░░░░] 6/10
      trigger: 카엘의 등장

🧑 카엘
   대화 스타일:
      tone: 차가운
      catchphrases: []
      forbidden_topics: []
   현재 감정:
      emotion: 냉담
      intensity: [███████░░░] 7/10
      trigger: None


## 3. 🗣️ TTS Voice Config 검증

> 음성 합성에 필요한 파라미터 (ElevenLabs, Google TTS 등)

In [6]:
if not dialogue_data:
    print("❌ 캐릭터 데이터 없음")
else:
    print("="*70)
    print("🗣️ TTS Voice Config (음성 합성 파라미터)")
    print("="*70)

    for name, data in dialogue_data.items():
        voice = data.get('voice_config', {})
        
        print(f"\n🧑 {name}")
        print(f"   base_pitch: {voice.get('base_pitch', 'N/A')}")
        print(f"   speaking_rate: {voice.get('speaking_rate', 'N/A')}")
        print(f"   stability: {voice.get('stability', 'N/A')}")
        print(f"   suggested_voice_id: {voice.get('suggested_voice_id', 'N/A')}")

🗣️ TTS Voice Config (음성 합성 파라미터)

🧑 아린
   base_pitch: medium
   speaking_rate: 1.1
   stability: 0.3
   suggested_voice_id: None

🧑 카엘
   base_pitch: low
   speaking_rate: 0.85
   stability: 0.3
   suggested_voice_id: None


## 4. 💬 Speech Samples 검증 (Few-Shot 프롬프팅용)

> 캐릭터 말투를 LLM에게 학습시키기 위한 예시 문장

In [7]:
if not dialogue_data:
    print("❌ 캐릭터 데이터 없음")
else:
    print("="*70)
    print("💬 Speech Samples (Few-Shot 프롬프팅용)")
    print("="*70)

    for name, data in dialogue_data.items():
        samples = data.get('speech_samples', [])
        
        print(f"\n🧑 {name}")
        if samples:
            for i, sample in enumerate(samples, 1):
                print(f"   {i}. \"{sample}\"")
        else:
            print("   (샘플 없음 - 직접 대사가 스토리에 없을 수 있음)")

💬 Speech Samples (Few-Shot 프롬프팅용)

🧑 아린
   1. "오랜만이군, 아린."
   2. "누구야?"
   3. "배신자와 할 말은 없어."

🧑 카엘
   1. "오랜만이군, 아린."
   2. "너를 찾고 있었어."
   3. "이번엔 네가 지는 거다."


## 5. 🎭 Dialogue Components 검증 (비언어적 표현 분리)

> 게임 엔진(Unity/Unreal)에서 텍스트와 애니메이션을 분리

In [8]:
if not dialogue_data:
    print("❌ 캐릭터 데이터 없음")
else:
    print("="*70)
    print("🎭 Dialogue Components (비언어적 표현 분리)")
    print("="*70)

    for name, data in dialogue_data.items():
        components = data.get('dialogue_components', [])
        
        print(f"\n🧑 {name}")
        if components:
            for comp in components:
                print(f"   clean_text: {comp.get('clean_text', 'N/A')}")
                print(f"   performance_guide: {comp.get('performance_guide', 'N/A')}")
                print(f"   anim_triggers: {comp.get('anim_triggers', [])}")
                print()
        else:
            print("   (비언어적 표현 없음)")

🎭 Dialogue Components (비언어적 표현 분리)

🧑 아린
   clean_text: 아린은 입술을 깨물며 전투 자세를 취했다. '배신자와 할 말은 없어.' 그녀의 심장이 빠르게 뛰기 시작했다.
   performance_guide: 한숨을 쉬며
   anim_triggers: ['ANIM_SIGH']


🧑 카엘
   (비언어적 표현 없음)


## 6. ⚠️ CRITICAL: 일시적 감정 검증

In [9]:
if not dialogue_data:
    print("❌ 캐릭터 데이터 없음")
else:
    print("="*70)
    print("⚠️ CRITICAL: 일시적 감정 검증")
    print("="*70)

    EXPECTED_EMOTIONS = {
        '아린': ['두려움', 'fear', '불안', 'anxious', '긴장'],
        '카엘': ['차가움', 'cold', '적대', '결의', '분노']
    }

    for name, expected in EXPECTED_EMOTIONS.items():
        data = dialogue_data.get(name, {})
        mood = data.get('current_mood', {})
        emotion = str(mood.get('emotion', '')).lower()
        
        print(f"\n{name}:")
        print(f"   current_mood.emotion: {mood.get('emotion')}")
        print(f"   intensity: {mood.get('intensity')}")
        print(f"   trigger: {mood.get('trigger')}")
        
        # Check if any expected emotion is present
        found = any(e.lower() in emotion for e in expected) if emotion else False
        if found or emotion:
            print(f"   ✅ 감정 추출됨")
        else:
            print(f"   ⚠️ 감정 없음 (예상: {expected})")

⚠️ CRITICAL: 일시적 감정 검증

아린:
   current_mood.emotion: 두려움
   intensity: 6
   trigger: 카엘의 등장
   ✅ 감정 추출됨

카엘:
   current_mood.emotion: 냉담
   intensity: 7
   trigger: None
   ✅ 감정 추출됨


## 7. Full JSON 출력

In [10]:
print("="*70)
print("📄 Full JSON Output (Production Ready)")
print("="*70)
if dialogue_data:
    print(json.dumps(dialogue_data, ensure_ascii=False, indent=2))
else:
    print("{}")
    print("\n❌ 데이터 없음")

📄 Full JSON Output (Production Ready)
{
  "아린": {
    "name": "아린",
    "dialogue": {
      "tone": "단호",
      "catchphrases": [],
      "forbidden_topics": [],
      "secret_keys": []
    },
    "current_mood": {
      "emotion": "두려움",
      "intensity": 6,
      "trigger": "카엘의 등장"
    },
    "voice_config": {
      "base_pitch": "medium",
      "speaking_rate": 1.1,
      "stability": 0.3,
      "suggested_voice_id": null
    },
    "speech_samples": [
      "오랜만이군, 아린.",
      "누구야?",
      "배신자와 할 말은 없어."
    ],
    "dialogue_components": [
      {
        "clean_text": "아린은 입술을 깨물며 전투 자세를 취했다. '배신자와 할 말은 없어.' 그녀의 심장이 빠르게 뛰기 시작했다.",
        "performance_guide": "한숨을 쉬며",
        "anim_triggers": [
          "ANIM_SIGH"
        ]
      }
    ]
  },
  "카엘": {
    "name": "카엘",
    "dialogue": {
      "tone": "차가운",
      "catchphrases": [],
      "forbidden_topics": [],
      "secret_keys": []
    },
    "current_mood": {
      "emotion": "냉담",
      "intensity": 7,
      "trigger

## 8. Production 체크리스트

In [11]:
print("="*70)
print("✅ Production 체크리스트")
print("="*70)

checks = []

# 1. 캐릭터 존재
if len(dialogue_data) >= 2:
    checks.append(("✅", "2+ characters extracted"))
elif len(dialogue_data) == 1:
    checks.append(("⚠️", f"Only 1 character (expected 2)"))
else:
    checks.append(("❌", f"Only {len(dialogue_data)} characters"))

if dialogue_data:
    # 2. voice_config
    has_voice = any(data.get('voice_config') for data in dialogue_data.values())
    if has_voice:
        checks.append(("✅", "TTS: voice_config generated"))
    else:
        checks.append(("❌", "TTS: Missing voice_config"))
    
    # 3. speech_samples
    has_samples = any(data.get('speech_samples') for data in dialogue_data.values())
    if has_samples:
        checks.append(("✅", "Few-Shot: speech_samples extracted"))
    else:
        checks.append(("⚠️", "Few-Shot: No speech_samples (may be OK if no direct dialogue)"))
    
    # 4. dialogue_components
    checks.append(("✅", "Game Engine: dialogue_components field present"))
    
    # 5. current_mood
    has_mood = any(data.get('current_mood', {}).get('emotion') for data in dialogue_data.values())
    if has_mood:
        checks.append(("✅", "Emotion: current_mood.emotion extracted"))
    else:
        checks.append(("⚠️", "Emotion: Missing current_mood.emotion"))

print()
for status, msg in checks:
    print(f"{status} {msg}")

print("\n" + "=" * 70)
passed = sum(1 for s, _ in checks if s == "✅")
total = len(checks)
print(f"결과: {passed}/{total} checks passed")

✅ Production 체크리스트

✅ 2+ characters extracted
✅ TTS: voice_config generated
✅ Few-Shot: speech_samples extracted
✅ Game Engine: dialogue_components field present
✅ Emotion: current_mood.emotion extracted

결과: 5/5 checks passed


## 9. 디버그 정보

In [12]:
print("="*70)
print("🔍 디버그 정보")
print("="*70)
print(f"\nResult keys: {result.keys()}")
print(f"Errors: {result.get('errors', [])}")
print(f"Messages: {result.get('messages', [])}")
print(f"Completed agents: {result.get('completed_agents', [])}")

🔍 디버그 정보

Result keys: dict_keys(['char_dialogue_mood', 'completed_agents', 'messages'])
Errors: []
Messages: [{'role': 'dialogue_mood_agent', 'content': 'Extracted 2 character dialogues/moods (production-ready)'}]
Completed agents: ['dialogue_mood']
